<a href="https://colab.research.google.com/github/12halima/Transport_Recommander/blob/main/Process_GTFS-OSM/Network_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
!ls "/content/drive/MyDrive"

GTFS_FINAL  HALIMA_ELBAHLOULI_CV_PFE-1.pdf  Screenshot_20251013-230531.jpg


In [2]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.3/455.3 MB 890.2 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 18.8 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.1.0-py2.py3-none-any.whl size=455986285 sha256=c3cf620e90a730c24367f096581e0835830f90ddf8c84985fa7cde2fa3596312
  Stored in directory: /root/.cache/pip/wheels/6b/9b/7c/2bea6ee44c4721d1af223365b8ab673a2db3ed4b759c12439d
Successfully built pyspark


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GTFS_Network_Creation") \
    .getOrCreate()

print("Spark version:", spark.version)


Spark version: 4.1.0


In [21]:
# ===============================
# 2. Définition des chemins GTFS
# ===============================

# Dossier principal dans Google Drive
BASE_PATH = "/content/drive/MyDrive/GTFS_FINAL"

# Dossier contenant les GTFS nettoyés par ville
CLEAN_PATH = f"{BASE_PATH}/GTFS_CLEAN"

# Fichier stop_times global (33+ millions de lignes)
STOP_TIMES_PATH = f"{BASE_PATH}/stop_times_final.csv"

print("BASE_PATH :", BASE_PATH)
print("CLEAN_PATH :", CLEAN_PATH)
print("STOP_TIMES_PATH :", STOP_TIMES_PATH)


BASE_PATH : /content/drive/MyDrive/GTFS_FINAL
CLEAN_PATH : /content/drive/MyDrive/GTFS_FINAL/GTFS_CLEAN
STOP_TIMES_PATH : /content/drive/MyDrive/GTFS_FINAL/stop_times_final.csv


In [22]:
import os

print("GTFS_FINAL existe :", os.path.exists(BASE_PATH))
print("GTFS_CLEAN existe :", os.path.exists(CLEAN_PATH))
print("stop_times_final.csv existe :", os.path.exists(STOP_TIMES_PATH))



GTFS_FINAL existe : True
GTFS_CLEAN existe : True
stop_times_final.csv existe : True


In [23]:
# ===============================
# 4. Chargement des fichiers GTFS par ville
# ===============================

import os

def load_gtfs_city(city_folder):
    """
    Charge stops, trips, routes pour une ville GTFS
    """
    city_path = os.path.join(CLEAN_PATH, city_folder)

    stops = spark.read.option("header", True).csv(f"{city_path}/stop_clean.txt")
    trips = spark.read.option("header", True).csv(f"{city_path}/trips_clean.txt")
    routes = spark.read.option("header", True).csv(f"{city_path}/routes_clean.txt")

    return stops, trips, routes


# Liste des villes disponibles
cities = [
    d for d in os.listdir(CLEAN_PATH)
    if os.path.isdir(os.path.join(CLEAN_PATH, d))
]

print("Villes détectées :", cities)


Villes détectées : ['Consorcio Regional de Transportes de Madrid Réseau de navettes CRTM (Red de Cercanías)', 'Junta de Extremadura (Bus du gouvernement régional d’Estrémadure)', 'FGV - Generalitat Valenciana Metro de Valencia', 'Avanza Grupo (Segovia city bus)', 'Empresa Municipal de Transports Urbans de Palma de Mallorca', 'Transports Municipaux D’Egara (TMESA) Terrassa bus urbain', 'Guaguas Municipales', 'Consorcio Regional de Transportes de Madrid CRTM Light Rail Network (Red de Metro Ligero)', 'City Council of Las Palmas de Gran Canaria (city bus)', 'Vectalia Movilidad (bus de la ville de Cáceres)', 'TIB Transports de les Illes Balears - CTM Transport Consortium of Mallorcam (Public transport on the island of Mallorca)', 'Alvarez Travelers Coaches', 'Avanza Grupo (VAC-124. Huesca-Lleida with branches)', 'Rafael Nadal Coaches', 'Costa Coaches', 'Xunta de Galicia Buses', 'Autocares Rías Baixas (Rías Baixas Coaches)', 'Transports Municipals del Gironés SAU (TMG) Girona city b

In [ ]:
# Exemple : première ville
city_name = cities[0]

stops_df, trips_df, routes_df = load_gtfs_city(city_name)

print("Ville :", city_name)
print("Stops :", stops_df.count())
print("Trips :", trips_df.count())
print("Routes :", routes_df.count())


Ville : Vectalia Movilidad (bus de la ville de Cáceres)
Stops : 237
Trips : 15260
Routes : 43


In [ ]:
stop_times.count()


31751871

In [ ]:
stop_times.show(10)


+-----------------+------------+--------------+-------+-------------+-------------+
|          trip_id|arrival_time|departure_time|stop_id|stop_sequence|source_folder|
+-----------------+------------+--------------+-------+-------------+-------------+
|2855_1_704_339306|     7:05:00|      07:05:00|      2|            0|            0|
|2855_1_704_339306|     7:08:00|      07:08:00|      3|            1|            0|
|2855_1_704_339306|     7:10:00|      07:10:00|      4|            2|            0|
|2855_1_704_339306|     7:12:00|      07:12:00|      5|            3|            0|
|2855_1_704_339306|     7:14:00|      07:14:00|      6|            4|            0|
|2855_1_704_339306|     7:15:00|      07:15:00|      7|            5|            0|
|2855_1_704_339306|     7:21:00|      07:21:00|      8|            6|            0|
|2855_1_704_339306|     7:22:00|      07:22:00|    240|            7|            0|
|2855_1_704_339306|     7:24:00|      07:24:00|    241|            8|       

In [24]:
stop_times = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(STOP_TIMES_PATH)

print("✅ stop_times chargé (schéma inféré automatiquement)")
stop_times.printSchema()

# Vérification rapide
stop_times.show(5)

✅ stop_times chargé (schéma inféré automatiquement)
root
 |-- trip_id: string (nullable = true)
 |-- arrival_time: timestamp (nullable = true)
 |-- departure_time: timestamp (nullable = true)
 |-- stop_id: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- pickup_type: integer (nullable = true)
 |-- drop_off_type: integer (nullable = true)
 |-- source_folder: string (nullable = true)

+-----------------+-------------------+-------------------+-------+-------------+-----------+-------------+--------------------+
|          trip_id|       arrival_time|     departure_time|stop_id|stop_sequence|pickup_type|drop_off_type|       source_folder|
+-----------------+-------------------+-------------------+-------+-------------+-----------+-------------+--------------------+
|2855_1_704_339306|2025-12-26 07:05:00|2025-12-26 07:05:00|      2|            0|          0|            0|Vectalia Movilida...|
|2855_1_704_339306|2025-12-26 07:08:00|2025-12-26 07:08:00|      3|    

In [25]:
from pyspark.sql.functions import col, broadcast
import os

OUTPUT_BASE = f"{BASE_PATH}/NETWORK_BASE_1"
os.makedirs(OUTPUT_BASE, exist_ok=True)

for city in cities:
    print(f"\n🚀 Traitement de la ville : {city}")
    city_path = os.path.join(CLEAN_PATH, city)

    try:
        # Charger GTFS de la ville
        stops_df, trips_df, routes_df = load_gtfs_city(city)

        # Sélection minimale pour réduire mémoire
        stops_df = stops_df.select("stop_id", "stop_name", "stop_lat", "stop_lon")
        trips_df = trips_df.select("trip_id", "route_id")
        routes_df = routes_df.select("route_id", "route_type", "route_long_name", "route_short_name")

        # Filtrer uniquement les lignes de stop_times correspondant à la ville
        stop_times_city = stop_times.filter(col("source_folder") == city).cache()

        # Jointure stop_times ↔ trips
        st_times_trips = stop_times_city.join(broadcast(trips_df), on="trip_id", how="inner")

        # Jointure avec stops
        st_times_trips_stops = st_times_trips.join(broadcast(stops_df), on="stop_id", how="inner")

        # Jointure avec routes (via route_id)
        network_base = st_times_trips_stops.join(broadcast(routes_df), on="route_id", how="left")

        # Sauvegarde par ville
        output_path = os.path.join(OUTPUT_BASE, city)
        network_base.repartition(50).write.mode("overwrite").parquet(output_path)

        stop_times_city.unpersist()
        print(f"✅ Ville {city} traitée et sauvegardée")

    except Exception as e:
        print(f"❌ Erreur pour la ville {city} : {e}")



🚀 Traitement de la ville : Consorcio Regional de Transportes de Madrid Réseau de navettes CRTM (Red de Cercanías)
✅ Ville Consorcio Regional de Transportes de Madrid Réseau de navettes CRTM (Red de Cercanías) traitée et sauvegardée

🚀 Traitement de la ville : Junta de Extremadura (Bus du gouvernement régional d’Estrémadure)
✅ Ville Junta de Extremadura (Bus du gouvernement régional d’Estrémadure) traitée et sauvegardée

🚀 Traitement de la ville : FGV - Generalitat Valenciana Metro de Valencia
✅ Ville FGV - Generalitat Valenciana Metro de Valencia traitée et sauvegardée

🚀 Traitement de la ville : Avanza Grupo (Segovia city bus)
✅ Ville Avanza Grupo (Segovia city bus) traitée et sauvegardée

🚀 Traitement de la ville : Empresa Municipal de Transports Urbans de Palma de Mallorca
✅ Ville Empresa Municipal de Transports Urbans de Palma de Mallorca traitée et sauvegardée

🚀 Traitement de la ville : Transports Municipaux D’Egara (TMESA) Terrassa bus urbain
✅ Ville Transports Municipa

In [26]:
OUTPUT_BASE = f"{BASE_PATH}/NETWORK_BASE_1"
city = cities[0]
df = spark.read.parquet(f"{OUTPUT_BASE}/{city}")

df.show(5, truncate=False)
df.printSchema()
print("Nombre de lignes :", df.count())


+--------+-------+-------+------------+--------------+-------------+-----------+-------------+-------------+---------+--------+--------+----------+---------------+----------------+
|route_id|stop_id|trip_id|arrival_time|departure_time|stop_sequence|pickup_type|drop_off_type|source_folder|stop_name|stop_lat|stop_lon|route_type|route_long_name|route_short_name|
+--------+-------+-------+------------+--------------+-------------+-----------+-------------+-------------+---------+--------+--------+----------+---------------+----------------+
+--------+-------+-------+------------+--------------+-------------+-----------+-------------+-------------+---------+--------+--------+----------+---------------+----------------+

root
 |-- route_id: string (nullable = true)
 |-- stop_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- arrival_time: timestamp (nullable = true)
 |-- departure_time: timestamp (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- picku

In [27]:
# Vérifier si des lignes sont complètement vides (toutes les colonnes sont nulles)
empty_rows = df.filter(
    (df["stop_id"].isNull()) &
    (df["trip_id"].isNull()) &
    (df["arrival_time"].isNull()) &
    (df["departure_time"].isNull()) &
    (df["stop_sequence"].isNull()) &
    (df["pickup_type"].isNull()) &
    (df["drop_off_type"].isNull()) &
    (df["source_folder"].isNull()) &
    (df["route_id"].isNull()) &
    (df["stop_name"].isNull()) &
    (df["stop_lat"].isNull()) &
    (df["stop_lon"].isNull()) &
    (df["route_type"].isNull()) &
    (df["route_short_name"].isNull()) &
    (df["route_long_name"].isNull())
)

# Afficher le nombre de lignes vides
empty_row_count = empty_rows.count()
print(f"Nombre de lignes complètement vides : {empty_row_count}")

# Optionnel: Afficher quelques exemples de lignes vides
empty_rows.select(
    "stop_id", "trip_id", "arrival_time", "departure_time",
    "route_id", "route_type", "route_short_name", "route_long_name",
    "stop_name", "stop_lat", "stop_lon"
).show(5, truncate=False)


Nombre de lignes complètement vides : 0
+-------+-------+------------+--------------+--------+----------+----------------+---------------+---------+--------+--------+
|stop_id|trip_id|arrival_time|departure_time|route_id|route_type|route_short_name|route_long_name|stop_name|stop_lat|stop_lon|
+-------+-------+------------+--------------+--------+----------+----------------+---------------+---------+--------+--------+
+-------+-------+------------+--------------+--------+----------+----------------+---------------+---------+--------+--------+



In [2]:
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, lead, unix_timestamp, lit, date_format
from pyspark.sql.window import Window

# ===============================
# Configuration des chemins
# ===============================
OUTPUT_BASE = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_BASE_1"  # Dossier contenant les données par ville
OUTPUT_EDGES_BASE = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_1"  # Dossier de sortie pour les edges
os.makedirs(OUTPUT_EDGES_BASE, exist_ok=True)

# Liste des villes (dossiers)
cities = [f for f in os.listdir(OUTPUT_BASE) if os.path.isdir(os.path.join(OUTPUT_BASE, f))]
print("Villes détectées :", cities)

# ===============================
# Chargement de toutes les villes
# ===============================
dfs = []
for city in cities:
    city_df = spark.read.parquet(f"{OUTPUT_BASE}/{city}") \
        .withColumn("city", lit(city))  # Ajouter le nom de la ville
    dfs.append(city_df)

# Combiner tous les DataFrames en un seul
df_all = reduce(DataFrame.unionByName, dfs)

# ===============================
# Création des edges (A → B)
# ===============================

# Sécurité : caster lat/lon en double
df_all = df_all.withColumn("stop_lat", col("stop_lat").cast("double")) \
               .withColumn("stop_lon", col("stop_lon").cast("double"))

# Fenêtre par trip (ordre des arrêts)
w = Window.partitionBy("trip_id").orderBy("stop_sequence")

# Récupérer les infos du stop suivant
edges = df_all \
    .withColumn("to_stop_id", lead("stop_id").over(w)) \
    .withColumn("to_lat", lead("stop_lat").over(w)) \
    .withColumn("to_lon", lead("stop_lon").over(w)) \
    .withColumn("to_arrival_time", lead("arrival_time").over(w)) \
    .withColumn("to_departure_time", lead("departure_time").over(w)) \
    .withColumn("from_stop_id", col("stop_id")) \
    .withColumn("from_lat", col("stop_lat")) \
    .withColumn("from_lon", col("stop_lon")) \
    .withColumn("from_arrival_time", col("arrival_time")) \
    .withColumn("from_departure_time", col("departure_time"))

# Supprimer les dernières lignes de chaque trip (pas de successeur)
edges = edges.filter(col("to_stop_id").isNotNull())

# Calcul du temps de trajet (secondes) basé sur arrival_time
edges = edges.withColumn(
    "travel_time_sec",
    unix_timestamp(col("to_arrival_time")) - unix_timestamp(col("from_arrival_time"))
)

# Calcul du temps de trajet basé sur departure_time (optionnel)
edges = edges.withColumn(
    "travel_time_departure_sec",
    unix_timestamp(col("to_departure_time")) - unix_timestamp(col("from_departure_time"))
)

# Calcul des deltas géographiques
edges = edges.withColumn("delta_lat", col("to_lat") - col("from_lat")) \
             .withColumn("delta_lon", col("to_lon") - col("from_lon"))

# Transformer les timestamps en heure HH:mm:ss
edges = edges \
    .withColumn("from_arrival_time", date_format(col("from_arrival_time"), "HH:mm:ss")) \
    .withColumn("from_departure_time", date_format(col("from_departure_time"), "HH:mm:ss")) \
    .withColumn("to_arrival_time", date_format(col("to_arrival_time"), "HH:mm:ss")) \
    .withColumn("to_departure_time", date_format(col("to_departure_time"), "HH:mm:ss"))

# Sélection finale
edges_final = edges.select(
    "city",
    "trip_id",
    "route_short_name",
    "route_long_name",
    "route_id",
    "route_type",
    "from_stop_id",
    "to_stop_id",
    "from_lat",
    "from_lon",
    "to_lat",
    "to_lon",
    "from_arrival_time",
    "from_departure_time",
    "to_arrival_time",
    "to_departure_time",
    "travel_time_sec",
    "travel_time_departure_sec",
    "delta_lat",
    "delta_lon"
)

print("✅ Edges créés pour toutes les villes")
edges_final.printSchema()
edges_final.limit(5).show(truncate=False)
print("Nombre total d'edges :", edges_final.count())


Villes détectées : ['Consorcio Regional de Transportes de Madrid Réseau de navettes CRTM (Red de Cercanías)', 'Junta de Extremadura (Bus du gouvernement régional d’Estrémadure)', 'FGV - Generalitat Valenciana Metro de Valencia', 'Avanza Grupo (Segovia city bus)', 'Empresa Municipal de Transports Urbans de Palma de Mallorca', 'Transports Municipaux D’Egara (TMESA) Terrassa bus urbain', 'Guaguas Municipales', 'Consorcio Regional de Transportes de Madrid CRTM Light Rail Network (Red de Metro Ligero)', 'City Council of Las Palmas de Gran Canaria (city bus)', 'Vectalia Movilidad (bus de la ville de Cáceres)', 'TIB Transports de les Illes Balears - CTM Transport Consortium of Mallorcam (Public transport on the island of Mallorca)', 'Alvarez Travelers Coaches', 'Avanza Grupo (VAC-124. Huesca-Lleida with branches)', 'Rafael Nadal Coaches', 'Costa Coaches', 'Xunta de Galicia Buses', 'Autocares Rías Baixas (Rías Baixas Coaches)', 'Transports Municipals del Gironés SAU (TMG) Girona city b

In [3]:
# Nombre de villes distinctes
num_cities = edges_final.select("city").distinct().count()
print("Nombre de villes distinctes :", num_cities)


Nombre de villes distinctes : 84


In [5]:
import os
from pyspark.sql.functions import col

# Dossier de sortie pour tous les edges (déjà créé)
os.makedirs(OUTPUT_EDGES_BASE, exist_ok=True)

# Récupérer la liste des villes présentes dans edges_final
cities_in_df = edges_final.select("city").distinct().rdd.flatMap(lambda x: x).collect()


# Sauvegarde par ville
for city_name in cities_in_df:
    # Créer un nom de dossier sûr (remplacer espaces et caractères spéciaux)
    safe_city_name = city_name.replace("/", "_").replace(" ", "_").replace("'", "_")

    # Filtrer les edges pour cette ville
    city_edges = edges_final.filter(col("city") == city_name)

    # Chemin de sortie pour cette ville
    output_path = os.path.join(OUTPUT_EDGES_BASE, safe_city_name)

    # ✅ Sauvegarde en Parquet avec repartition pour éviter OOM
    city_edges.repartition(50).write.mode("overwrite").parquet(output_path)

    # ✅ Utiliser limit() pour éviter un count trop lourd
    preview = city_edges.limit(5).toPandas()
    print(f"✅ Edges de {city_name} sauvegardés dans {output_path}")
    print(preview)

print("🎉 Toutes les villes ont été sauvegardées !")


ConnectionRefusedError: [Errno 111] Connection refused